In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Custom AlltoAllV with NVSHMEM and NCCL

## Overview

A custom collective starts with a data contract, not a transport choice. This notebook uses an irregular `AlltoAllV` to show how the same contract maps to NVSHMEM and NCCL device APIs. NVSHMEM selects a direct or network route at runtime in one kernel. NCCL selects an algorithm that matches the available communication scope: LSA, full GIN, or LSA plus railed GIN.

By the end, you will be able to:

- define the counts and offsets that make an `AlltoAllV` genuinely irregular;
- validate an out-of-place collective without accidentally accepting a fixed-size all-to-all;
- distinguish remote completion from local source-buffer reuse;
- map the same transfer plan to NVSHMEM direct/network routes and NCCL LSA/GIN paths; and
- compare implementations without mistaking a topology change for an algorithmic improvement.

The executable Python cells model the data contract only. The CUDA examples are intentionally presented as code excerpts: running the real kernels requires Linux, multiple NVIDIA GPUs, MPI, and current NVSHMEM or NCCL device-API installations.

---
## 1. The collective contract

For source rank `r` and destination rank `d`, `send_counts[r][d]` values beginning at `send_offsets[r][d]` belong at `recv_offsets[d][r]` on rank `d`. The matching receive count is therefore `recv_counts[d][r] = send_counts[r][d]`. Counts, source offsets, and destination offsets may all differ by rank pair.

```text
source rank r                              destination rank d

send buffer                                receive buffer
+-----+----------+----+                    +----+----------+-----+
| gap | payload  | gap|  ---------------->  | gap| payload  | gap |
+-----+----------+----+                    +----+----------+-----+
        send_offsets[r][d]                     recv_offsets[d][r]
```

The model below deliberately leaves gaps in both buffers. Its validation checks payload values and confirms that no transfer overwrote an untouched gap.

In [ ]:
def align(value, alignment):
    return (value + alignment - 1) // alignment * alignment


def make_offsets(counts, alignment=4, gap=3):
    cursor = 0
    offsets = []
    for count in counts:
        cursor = align(cursor + gap, alignment)
        offsets.append(cursor)
        cursor += count
    return offsets, cursor + gap


def make_send_buffer(source, counts, offsets, capacity):
    buffer = [None] * capacity
    for destination, count in enumerate(counts):
        for index in range(count):
            buffer[offsets[destination] + index] = (source, destination, index)
    return buffer


def alltoallv_model(send_counts, send_offsets, recv_offsets, send_buffers, recv_capacities):
    recv_buffers = [[None] * capacity for capacity in recv_capacities]
    for source, row in enumerate(send_counts):
        for destination, count in enumerate(row):
            send_begin = send_offsets[source][destination]
            recv_begin = recv_offsets[destination][source]
            recv_buffers[destination][recv_begin:recv_begin + count] = (
                send_buffers[source][send_begin:send_begin + count]
            )
    return recv_buffers


def validate_alltoallv(send_counts, recv_offsets, recv_buffers):
    written = [set() for _ in recv_buffers]
    for source, row in enumerate(send_counts):
        for destination, count in enumerate(row):
            recv_begin = recv_offsets[destination][source]
            for index in range(count):
                assert recv_buffers[destination][recv_begin + index] == (source, destination, index)
                written[destination].add(recv_begin + index)
    for destination, buffer in enumerate(recv_buffers):
        untouched = set(range(len(buffer))) - written[destination]
        assert all(buffer[index] is None for index in untouched)


send_counts = [[2, 0, 3], [1, 4, 0], [0, 2, 1]]
send_layouts = [make_offsets(row) for row in send_counts]
send_offsets = [offsets for offsets, _ in send_layouts]
send_buffers = [
    make_send_buffer(source, row, offsets, capacity)
    for source, (row, (offsets, capacity)) in enumerate(zip(send_counts, send_layouts))
]
recv_counts = [list(column) for column in zip(*send_counts)]
recv_layouts = [make_offsets(row) for row in recv_counts]
recv_offsets = [offsets for offsets, _ in recv_layouts]
recv_capacities = [capacity for _, capacity in recv_layouts]
recv_buffers = alltoallv_model(
    send_counts, send_offsets, recv_offsets, send_buffers, recv_capacities
)
validate_alltoallv(send_counts, recv_offsets, recv_buffers)
print("PASS: irregular counts, non-contiguous offsets, and untouched gaps")

### Invariants that every implementation must preserve

- Each destination sees exactly the payload selected by each source's count and offset.
- A zero-count pair moves no data and contributes no expected completion event.
- The receiver chooses its own layout, so senders need the advertised receive offsets before issuing the transfer.
- A receiver may read a segment only after its payload is visible.
- A sender may reuse its source segment only after its own transport has completed the source-side operation.

The last two invariants sound similar, but they are different facts. A remote signal can make a destination buffer safe to read without making the sender's buffer safe to reuse.

---
## 2. One data contract, different API responsibilities

| Decision | NVSHMEM | NCCL device APIs |
| --- | --- | --- |
| Select a route | A single kernel calls `nvshmem_ptr` for each target. A non-null pointer takes the direct path; an unmapped peer takes the network path. | The application selects an implementation for the available scope: LSA, full GIN, or LSA plus railed GIN. |
| Move a direct-peer segment | A block-scoped nonblocking put with a signal. | CUDA threads copy through an LSA pointer into a registered peer window. |
| Move a network segment | A QP-specific nonblocking put with a signal. | A full-GIN put can target any world peer. Railed GIN writes an ingress inbox, followed by an LSA scatter. |
| Tell the receiver data is ready | The attached signal orders its payload before the signal update. | An LSA barrier publishes direct stores. A GIN weak signal covers the put that carries it. |
| Reuse the source range | `quiet` completes the sender's outstanding default and custom QP operations. | The LSA completion barrier closes the direct path; `gin.flush` completes the issuing GIN context. |

The important design choice is not which library is better in the abstract. It is which mechanism can address the peers in the placement while preserving the same plan and completion invariants.

---
## 3. NVSHMEM: choose the route inside one collective kernel

NVSHMEM can keep one implementation across direct and network placements. A setup phase exchanges byte counts, receiver-selected offsets, and the number of signal slots each sender will produce. That last exchange is necessary because `nvshmem_ptr` describes the sender's outgoing accessibility; it cannot be inferred from the receiver's opposite-direction query.

The payload phase can then assign destination/chunk pairs to CTAs. Short direct chunks create parallel work for directly mapped peers, while larger network chunks avoid turning a large remote message into too many operations.

```cpp
void* direct = nvshmem_ptr(remote_receive_address, destination);
if (direct != nullptr) {
  nvshmemx_putmem_signal_nbi_block(remote_receive_address, source, bytes,
                                      signal_slot, epoch, NVSHMEM_SIGNAL_SET, destination);
} else if (threadIdx.x == 0) {
  auto qp = network_qps[(destination + chunk) % network_qp_count];
  nvshmemx_qp_uint_put_signal_nbi(remote_values, source_values, value_count,
                                   signal_slot, epoch, NVSHMEM_SIGNAL_SET, destination, qp);
}
```

A cooperative launch makes the producer and completion phases safe to fuse. One controller CTA performs the cross-PE NVSHMEM barrier; CUDA grid barriers only coordinate CTAs on the local GPU. The controller waits until every producer CTA has issued its chunks, completes QP activity with the appropriate `quiet` scope, then waits for the route-plan signal counts. The cooperative grid must be concurrently resident, otherwise an early waiting CTA could prevent a producer CTA from running.

```text
one-time plan: counts all-to-all -> receive offsets -> sender-selected signal counts

each call: controller block barrier -> local grid release -> producers issue chunks
           -> local grid handoff -> controller quiets QPs -> waits for signals
```

Signals and `quiet` deliberately answer different questions: signals protect the receiver, while `quiet` protects the sender.

---
## 4. NCCL: select an algorithm for the communication scope

### LSA: direct stores inside one LSA domain

When every rank is in one LSA domain, the simplest algorithm is a direct copy. Each CTA takes an aligned slice of each rank-pair segment, translates the LSA-team peer to its world rank for plan lookup, and writes through `ncclGetLsaPointer`. There is no per-message staging buffer or signal.

```cpp
barrier.sync(ncclCoopCta(), cuda::memory_order_acquire);
for (int lsa_peer = 0; lsa_peer < lsa_team.nRanks; ++lsa_peer) {
  int world_peer = ncclTeamRankToWorld(dev_comm, lsa_team, lsa_peer);
  auto entry = plan[world_peer];
  auto source = ncclGetLocalPointer(send_window, entry.send_offset_bytes);
  auto destination = ncclGetLsaPointer(recv_window, entry.remote_recv_offset_bytes, lsa_peer);
  copy_this_cta_slice(source, destination, entry.count);
}
barrier.sync(ncclCoopCta(), cuda::memory_order_acq_rel);
```

The entry barrier prevents peer stores from beginning before every rank joins the collective. The completion barrier publishes the stores and returns only after the direct collective phase is complete. `ncclGetLsaPointer` receives an LSA-team rank, not a world rank; treating those rank spaces as interchangeable is a common correctness bug.

### Full GIN: GPU-initiated network puts

Full GIN lets a kernel issue a put to an arbitrary world peer's registered receive window. CTAs shard rank-pair messages, rotate route assignment to spread work over GIN contexts, and attach a weak signal to each non-empty put. The matching receiver CTA knows whether it expects an inbound shard from the plan.

```cpp
auto before = gin.readSignal(signal);
barrier.sync(ncclCoopCta(), cuda::memory_order_acquire, ncclGinFenceLevel::None);
if (threadIdx.x == 0 && shard.count != 0) {
  gin.put(ncclTeamWorld(dev_comm), peer, recv_window, recv_offset_bytes,
          send_window, send_offset_bytes, shard.bytes, ncclGin_WeakSignalInc{signal});
}
gin.waitSignal(ncclCoopCta(), signal, before + expected_incoming);
gin.flush(ncclCoopCta());
barrier.sync(ncclCoopCta(), cuda::memory_order_release, ncclGinFenceLevel::None);
```

A weak signal makes the payload of its own put visible before the receiver observes the increment. `gin.flush` is local: it makes the issuing CTA's source use complete, but it does not notify a peer. The final barrier is the collective rendezvous after both facts hold.

### LSA plus railed GIN: one inbox stage when rails are constrained

Railed GIN connects matching LSA ranks across domains; it cannot directly address every rank in a remote domain. A cross-domain message therefore lands in a fixed inbox slot on the matching ingress GPU and is scattered through an LSA pointer to its final local destination. Same-domain traffic still uses LSA directly.

```text
source domain                              destination domain

GPU 0 ===== rail 0 =====> GPU 0 inbox ---- LSA ----> final local GPU
GPU 1 ===== rail 1 =====> GPU 1 inbox ---- LSA ----> final local GPU
```

The inbox is an algorithmic consequence of the rail topology, not an implementation accident. It adds one local scatter for cross-domain data, so report its traffic and time separately when diagnosing performance.

---
## 5. Choose from topology and completion requirements

| Placement or goal | Natural implementation | Why |
| --- | --- | --- |
| One directly addressable LSA domain | NCCL LSA | Direct vectorized stores through peer windows with LSA barriers. |
| Arbitrary remote world peers | NCCL full GIN | Each CTA can put a shard into a named peer window and use a weak signal for that shard. |
| Several LSA domains with matching rails only | NCCL LSA plus railed GIN | The inbox plus LSA scatter bridges the difference between a rail peer and the final world-rank destination. |
| One source implementation spanning direct and network routes | NVSHMEM | `nvshmem_ptr` selects the direct route per target while QP-specific puts handle unmapped peers. |

Every case still needs the same plan. What changes is the way the kernel obtains a destination address, exposes enough parallel work, and proves completion.

---
## 6. Benchmark the implementation, not a changed workload

Keep the off-diagonal payload, rank placement, counts, offsets, warmup, and measured iterations fixed when comparing implementations. Record these configuration facts with every result:

- number of direct and network routes;
- chunk or shard size, CTA count, and thread count;
- NVSHMEM QP count or NCCL GIN context and queue-depth settings;
- logical non-self bytes divided by the slowest-rank time; and
- direct and cross-domain payload rates separately.

A faster result after changing the number of direct peers is a topology change, not evidence that an API implementation became faster. Likewise, a mixed direct/network logical rate is not an InfiniBand bandwidth number.

---
## 7. Challenge

Take the Python plan above and change one `send_counts[source][destination]` entry to zero while keeping all offsets unchanged. Before running it, predict which receive segment becomes empty and why the corresponding route must not create a signal expectation. Then restore the count and deliberately swap two receive offsets: the validator should fail because payload placement, not just payload volume, is part of the contract.

For a hardware implementation, repeat the same exercise with the source tutorial's `sparse` pattern. Verify the full receive buffer and its untouched gaps before interpreting timing output.

---
## 8. Run the complete implementations

The accompanying [topology-aware AlltoAllV tutorial](https://github.com/NVIDIA/hoti-gpu-comms-tutorial/tree/main/09-alltoallv) contains starter and solved CUDA sources for NVSHMEM, NCCL LSA, NCCL GIN, and NCCL LSA plus railed GIN. It provides the full device setup, correctness validation, build recipes, and topology-specific launch guidance.

Use an environment with matching NCCL headers and runtime for the NCCL device APIs. For NVSHMEM, use a current installation with explicit-QP support and create QPs collectively. A generic single-GPU notebook runtime cannot validate the remote completion behavior described here.

---
## Conclusion

Custom collectives are easiest to reason about when the data contract remains stable across implementations. NVSHMEM makes route selection part of one kernel and separates remote signals from local quiet. NCCL makes the communication scope explicit: LSA for directly addressable peers, GIN for remote puts, and an inbox plus LSA scatter when rails constrain the peer graph. Once the plan, completion facts, and measurement denominator stay fixed, the API differences become concrete engineering choices rather than folklore.

---
## References

- [Topology-aware AlltoAllV tutorial](https://github.com/NVIDIA/hoti-gpu-comms-tutorial/tree/main/09-alltoallv)
- [NVSHMEM signal operations](https://docs.nvidia.com/nvshmem/api/latest/gen/api/signal.html)
- [NVSHMEM QP APIs](https://docs.nvidia.com/nvshmem/api/latest/gen/api/qp.html)
- [NVSHMEM collective launch](https://docs.nvidia.com/nvshmem/api/latest/api/launch.html)
- [NCCL device API](https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/usage/deviceapi.html)
- [NCCL device memory and LSA pointers](https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/api/device_memory.html)
- [NCCL GIN](https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/api/device_gin.html)